# 🏆 CATIA V5 Solar Score Gap ≥ 2 고격차 12개 질문 전용 Qwen 0.5B 성능 비교 및 BERTScore 평가

## 📌 개요 및 평가 목적
본 노트북은 Upstage Solar 평가관 점수 차이(`solar_score_gap = RAG 점수 - Direct 점수`)가 **2점 이상 벌어진 최우수 고격차 질문 12개**([eval/CATIA_RAG_Solar_Gap_2_Plus_Questions.txt](file:///c:/KDT_14/%5B11%5DTransformer/project/team-03-project/eval/CATIA_RAG_Solar_Gap_2_Plus_Questions.txt))를 대상으로 **Qwen 2.5 0.5B 로컬 모델**의 답변 성능을 입증합니다.

**Direct LLM(RAG Off)의 환각 문장**과 **RAG LLM(RAG On)의 정확한 답변 문장**을 1:1 대조하고, **Upstage Solar 1~5점 자체 평가** 및 **BERTScore (Precision, Recall, F1)** 지표를 계산하여 RAG 적용에 따른 압도적인 성능 향상을 검증합니다.

--- 
### 📊 평가 문항 및 모델 정보
- **평가 대상 질문**: `solar_score_gap >= 2`를 만족하는 선별 질문 12개
- **추론 모델**: Qwen 2.5 0.5B (`Qwen/Qwen2.5-0.5B-Instruct` 100% 로컬 PyTorch)
- **평가관 모델**: Upstage Solar Pro (`solar-pro` via Upstage API)
- **정량적 지표**: Upstage Solar Score (1~5점) 및 BERTScore F1 (+ΔF1)

In [1]:
# 1. 환경 설정 및 주요 모듈 임포트
import os
import sys
import gc
import json
import pandas as pd
import torch
from pathlib import Path
from bert_score import score as bert_score_compute
from IPython.display import HTML, display
from langchain_openai import ChatOpenAI

# 프로젝트 루트 sys.path 추가
PROJECT_ROOT = Path(".").resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT / ".." / "src").exists():
    PROJECT_ROOT = (PROJECT_ROOT / "..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rag_chain import RAGPipeline
from src.config import DATA_DIR, CHROMA_DB_DIR

# Upstage API Key 설정
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY", "up_UFLAwDwDhV9YXWlUKU6CI3piyvp9q")
os.environ["UPSTAGE_API_KEY"] = UPSTAGE_API_KEY

print(f"[System] PyTorch CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[System] Active GPU: {torch.cuda.get_device_name(0)}")
print(f"[System] Upstage API Key Loaded: {UPSTAGE_API_KEY[:8]}...")


c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[System] PyTorch CUDA Available: True
[System] Active GPU: NVIDIA GeForce RTX 4070 Laptop GPU
[System] Upstage API Key Loaded: up_UFLAw...


In [2]:
# 2. Solar Gap ≥ 2 필터링 질문 데이터셋 로딩
auto_csv_path = PROJECT_ROOT / "eval" / "CATIA_RAG_High_Gap_Auto_Questions.csv"
txt_gap2_path = PROJECT_ROOT / "eval" / "CATIA_RAG_Solar_Gap_2_Plus_Questions.txt"

if not auto_csv_path.exists():
    raise FileNotFoundError(f"{auto_csv_path} 파일이 존재하지 않습니다.")

df_auto = pd.read_csv(auto_csv_path)
# solar_score_gap >= 2 필터링
df_gap2 = df_auto[df_auto["solar_score_gap"] >= 2].copy().reset_index(drop=True)
df_gap2["id"] = [f"Q{i+1:02d}" for i in range(len(df_gap2))]

print(f"[Dataset] Successfully loaded {len(df_gap2)} questions with solar_score_gap >= 2:")
display(df_gap2[["id", "question", "reference_answer", "solar_score_gap"]])


[Dataset] Successfully loaded 12 questions with solar_score_gap >= 2:


,id,question,reference_answer,solar_score_gap
0,Q01,CATIA V5에서 Specification Tree를 실시간으로 숨기거나 표시하는...,F3,3
1,Q02,CATIA V5에서 'Part Design Workbench'를 즉시 활성화하기 위...,Ctrl+W,3
2,Q03,CATIA V5에서 'Reference Elements (Extended)' 기능을...,H와 V 값은 모두 0으로 지정해야 합니다.,2
3,Q04,CATIA V5의 Tools 툴바에서 문서의 업데이트가 필요한 엔티티가 있을 때 활...,Update All,2
4,Q05,CATIA V5에서 'Navigation Mode' 메뉴를 활성화하기 위해 반드시 ...,Snap to Point,2
5,Q06,"CATIA V5에서 모든 툴바와 도구가 화면상에서 사라진 경우, 작업 영역 빈 공간...",해당 팝업 창에서 '툴 및 툴바 표시' 옵션에 해당하는 체크박스를 선택해야 합니다....,2
6,Q07,"CATIA V5에서 Shell 기능을 적용할 때, 특정 곡면에 대해 두께 값을 지정...",Shell 대화상자의 'Limits' 탭에서 'Multi-Pad limits' 옵션...,2
7,Q08,CATIA V5에서 'Specification Tree'를 실시간으로 숨기거나 다시...,F3,3
8,Q09,CATIA V5에서 'View Mode' 도구 아이콘의 오른쪽 하단에 표시된 화살표...,Back View,4
9,Q10,CATIA V5에서 새 파트 생성 시 축 시스템(Axis System)을 즉시 생성...,new part section,2


In [3]:
# 3. Qwen 2.5 0.5B 답변 생성 (Direct LLM vs RAG LLM)
MODEL_NAME = "Qwen 2.5 0.5B"
REPO_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"==================================================")
print(f"[Experiment] Initializing Local Qwen Model: {MODEL_NAME} ({REPO_ID})")
print(f"==================================================")

pipeline = RAGPipeline(model_name=REPO_ID, mode="local")

direct_answers = []
rag_answers = []
source_citations = []

for idx, row in df_gap2.iterrows():
    qid = row["id"]
    question = row["question"]
    print(f" [{idx+1}/{len(df_gap2)}] Generating Answers for {qid}...")
    
    # 1) Direct LLM (RAG Off)
    ans_direct = pipeline.answer_direct(question)
    direct_answers.append(ans_direct)
    
    # 2) RAG LLM (RAG On)
    res_rag = pipeline.answer_rag(question)
    ans_rag = res_rag.get("answer", "")
    sources = res_rag.get("source_pages", [])
    
    rag_answers.append(ans_rag)
    source_citations.append(", ".join(sources) if sources else "매뉴얼 문맥 참조")

print(f"\n[Experiment] Answer Generation Completed for {len(df_gap2)} High-Gap Questions!")


[Experiment] Initializing Local Qwen Model: Qwen 2.5 0.5B (Qwen/Qwen2.5-0.5B-Instruct)
[LLMFactory] Initializing LLM 'Qwen/Qwen2.5-0.5B-Instruct' in mode='local'...
[LLMFactory Local] Loading model 'Qwen/Qwen2.5-0.5B-Instruct' on device 'cuda'...


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Device set to use cuda:0


[VectorStore] Initializing Local Embeddings 'sentence-transformers/all-MiniLM-L6-v2'...
[VectorStore] Loading existing Chroma database from: C:\KDT_14\[11]Transformer\project\team-03-project\vect\chroma_db_multimodal


C:\KDT_14\[11]Transformer\project\team-03-project\src\vector_store.py:52: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


 [1/12] Generating Answers for Q01...


...

 [11/12] Generating Answers for Q11...


c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-p

 [12/12] Generating Answers for Q12...


c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-p


[Experiment] Answer Generation Completed for 12 High-Gap Questions!


In [4]:
# 4. Upstage Solar LLM-as-a-Judge 자체 평가 수행
solar_judge = ChatOpenAI(
    model="solar-pro",
    openai_api_key=UPSTAGE_API_KEY,
    openai_api_base="https://api.upstage.ai/v1/solar",
    temperature=0.0
)

SOLAR_EVAL_PROMPT = """당신은 CATIA V5 전문 CAD 평가관(LLM-as-a-Judge)입니다.
전문적인 CATIA V5 CAD 지식에 기반하여, 아래 질문에 대해 AI 모델이 생성한 답변의 정확성을 1점부터 5점까지 점수로 직접 평가해주세요.

[평가 점수 기준]
- 5점 (맞음 / 매우 정확함): 질문에서 요구한 CATIA V5의 정확한 기능명, 키보드 단축키, 마우스 조작법 및 절차가 오류 없이 명확하게 설명됨.
- 4점 (부분 맞음): 핵심 기능명이나 단축키가 대체로 맞으나 설명이 약간 부족하거나 사소한 미흡함이 있음.
- 3점 (보통 / 애매함): 일반적인 CAD 개념 언급은 있으나 질문에서 요청한 핵심 단축키나 명칭이 누락되어 모호함.
- 2점 (틀림 / 불일치): 엉뚱한 용어를 언급하거나 다른 CAD 기능과 혼동하여 오답에 가까움.
- 1점 (전혀 아님 / 환각): 존재하지 않는 단축키/기능을 지어내거나(환각), 무응답 또는 질문과 전혀 상관없는 오답.

[질문]: {question}
[모델 생성 답변]: {generated_answer}

반드시 아래 JSON 형식으로만 응답하세요:
{{
  "score": <1부터 5 사이 정수>,
  "reason": "<한 줄 평가 요약>"
}}"""

solar_direct_scores = []
solar_direct_reasons = []
solar_rag_scores = []
solar_rag_reasons = []

def evaluate_with_solar(q, ans):
    try:
        prompt = SOLAR_EVAL_PROMPT.format(question=q, generated_answer=ans)
        res = solar_judge.invoke(prompt)
        text = res.content.strip()
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0].strip()
        elif "```" in text:
            text = text.split("```")[1].split("```")[0].strip()
        data = json.loads(text)
        score = int(data.get("score", 1))
        reason = str(data.get("reason", ""))
        return min(max(score, 1), 5), reason
    except Exception:
        return 1, "평가 예외 발생"

for idx, row in df_gap2.iterrows():
    qid = row["id"]
    q = row["question"]
    print(f" [{idx+1}/{len(df_gap2)}] Solar Judge Evaluating {qid}...")
    
    s_dir, r_dir = evaluate_with_solar(q, direct_answers[idx])
    s_rag, r_rag = evaluate_with_solar(q, rag_answers[idx])
    
    solar_direct_scores.append(s_dir)
    solar_direct_reasons.append(r_dir)
    solar_rag_scores.append(s_rag)
    solar_rag_reasons.append(r_rag)

print("\n[Upstage Evaluator] Upstage Solar Evaluation Completed!")


 [1/12] Solar Judge Evaluating Q01...
 [2/12] Solar Judge Evaluating Q02...
 [3/12] Solar Judge Evaluating Q03...
 [4/12] Solar Judge Evaluating Q04...
 [5/12] Solar Judge Evaluating Q05...
 [6/12] Solar Judge Evaluating Q06...
 [7/12] Solar Judge Evaluating Q07...
 [8/12] Solar Judge Evaluating Q08...
 [9/12] Solar Judge Evaluating Q09...
 [10/12] Solar Judge Evaluating Q10...
 [11/12] Solar Judge Evaluating Q11...
 [12/12] Solar Judge Evaluating Q12...

[Upstage Evaluator] Upstage Solar Evaluation Completed!


## 👁️ 5. 생성 문장 1:1 Side-by-Side 대조 시각화 표 (Upstage Solar 1~5점 표시)
Solar Score Gap ≥ 2 질문별 **표준 정답(Ground Truth)**과 **Qwen 0.5B Direct LLM 답변** vs **RAG 적용 답변**을 1:1 대조하고, 셀 상단에 **Upstage Solar 1~5점 뱃지 및 채점 사유**를 표시한 시각화 표입니다.

In [5]:
# 5. HTML Side-by-Side 대조 표 생성 및 렌더링
score_labels = {
    5: ("맞음 (5/5)", "#15803d", "#dcfce7"),
    4: ("부분 맞음 (4/5)", "#0369a1", "#e0f2fe"),
    3: ("보통 (3/5)", "#b45309", "#fef3c7"),
    2: ("틀림 (2/5)", "#c2410c", "#ffedd5"),
    1: ("전혀 아님/환각 (1/5)", "#b91c1c", "#fee2e2")
}

css_style = """
<style>
    .eval-table {
        width: 100%;
        border-collapse: collapse;
        font-family: 'Segoe UI', Malgun Gothic, sans-serif;
        font-size: 13px;
        margin-top: 10px;
    }
    .eval-table th {
        background-color: #0f172a;
        color: #ffffff;
        text-align: center;
        padding: 10px;
        border: 1px solid #475569;
        font-weight: 600;
    }
    .eval-table td {
        padding: 10px 12px;
        border: 1px solid #cbd5e1;
        vertical-align: top;
        line-height: 1.5;
        word-break: break-word;
    }
    .col-gt { background-color: #f0fdf4; color: #166534; font-weight: 600; }
    .col-direct { background-color: #fafafa; color: #1e293b; padding: 8px; border-radius: 4px; border: 1px solid #e2e8f0; }
    .col-rag { background-color: #f0f9ff; color: #0369a1; padding: 8px; border-radius: 4px; border: 1px solid #bae6fd; font-weight: 500; }
    .badge { display: inline-block; padding: 3px 8px; border-radius: 4px; font-weight: bold; font-size: 11px; margin-bottom: 6px; }
    .badge-solar { display: inline-block; padding: 3px 8px; border-radius: 4px; font-weight: bold; font-size: 11px; margin-bottom: 6px; border: 1px solid; }
    .badge-direct { background-color: #fda4af; color: #881337; }
    .badge-rag { background-color: #93c5fd; color: #1e3a8a; }
</style>
"""

html_table = css_style + f"""
<div style="overflow-x: auto; border: 1px solid #cbd5e1; border-radius: 6px;">
    <table class="eval-table">
        <thead>
            <tr>
                <th style="width: 50px;">ID</th>
                <th style="width: 200px;">질문 (Question)</th>
                <th style="width: 240px;">표준 정답 (Ground Truth)</th>
                <th style="width: 330px;">{MODEL_NAME} Direct LLM (RAG Off)</th>
                <th style="width: 330px;">{MODEL_NAME} RAG LLM (RAG On)</th>
            </tr>
        </thead>
        <tbody>
"""

for idx, row in df_gap2.iterrows():
    qid = str(row.get("id", ""))
    q = str(row.get("question", ""))
    gt = str(row.get("reference_answer", ""))
    ans_dir = direct_answers[idx]
    ans_rag = rag_answers[idx]
    src = source_citations[idx]
    
    s_dir = solar_direct_scores[idx]
    r_dir = solar_direct_reasons[idx]
    lbl_dir, col_dir, bg_dir = score_labels.get(s_dir, ("평가전", "#000", "#fff"))
    
    s_rag = solar_rag_scores[idx]
    r_rag = solar_rag_reasons[idx]
    lbl_rag, col_rag, bg_rag = score_labels.get(s_rag, ("평가전", "#000", "#fff"))
    
    html_table += f"""
            <tr>
                <td style="text-align: center; font-weight: bold;">{qid}</td>
                <td><b>{q}</b></td>
                <td class="col-gt">{gt}</td>
                <td>
                    <span class="badge badge-direct">[RAG Off / Direct LLM]</span><br/>
                    <span class="badge-solar" style="background-color: {bg_dir}; color: {col_dir}; border-color: {col_dir};">⭐ Upstage Solar: {lbl_dir}</span><br/>
                    <small style="color: #475569;">💬 {r_dir}</small>
                    <div class="col-direct" style="margin-top: 6px;">{ans_dir}</div>
                </td>
                <td>
                    <span class="badge badge-rag">[RAG On / 매뉴얼 참조]</span><br/>
                    <span class="badge-solar" style="background-color: {bg_rag}; color: {col_rag}; border-color: {col_rag};">⭐ Upstage Solar: {lbl_rag}</span><br/>
                    <small style="color: #475569;">💬 {r_rag}</small>
                    <div class="col-rag" style="margin-top: 6px;">{ans_rag}</div><br/>
                    <small style="color: #475569;">📍 <b>참조 근거:</b> {src}</small>
                </td>
            </tr>
    """

html_table += """
        </tbody>
    </table>
</div>
"""

display(HTML(html_table))


ID,질문 (Question),표준 정답 (Ground Truth),Qwen 2.5 0.5B Direct LLM (RAG Off),Qwen 2.5 0.5B RAG LLM (RAG On)
Q01,CATIA V5에서 Specification Tree를 실시간으로 숨기거나 표시하는 기능을 수행하는 전용 토글 키는 무엇인가요?,F3,"[RAG Off / Direct LLM] ⭐ Upstage Solar: 틀림 (2/5) 💬 존재하지 않는 '터치 키(T/F)'라는 잘못된 용어를 사용했으며, 실제 단축키인 'Ctrl+Shift+T'를 언급하지 못해 오답에 가까움 CATIA V5에서 Specification Tree를 실시간으로 숨기거나 표시하는 기능을 제공하기 위해 사용되는 토글 키는 다음과 같습니다: 1. T: 터치 키 (T) 2. F: 터치 키 (F) 이 두 토글 키를 동시에 누르면, CATIA V5의 Specification Tree가 실시간으로 숨겨질 수 있습니다. 또한, 이 토글 키는 CATIA V5의 기본 키보드와 동일하게 작동하며, 터치 키를 사용하여 원하는 위치에 터치하면 해당 위치로 터치 키를 누르면 CATIA V5의 Specification Tree가 실시간으로 숨겨질 수 있습니다. 이러한 기능은 CATIA V5의 실시간 툴바와 관련되어 있으며, 사용자가 터치 키를 사용하여 원하는 위치에 터치하면 CATIA V5의 Specification Tree가 실시간으로 숨겨지는 것을 확인할 수 있습니다.","[RAG On / 매뉴얼 참조] ⭐ Upstage Solar: 맞음 (5/5) 💬 CATIA V5에서 Specification Tree를 토글하는 정확한 단축키(Ctrl + Shift + T)를 오류 없이 명시함. CATIA V5에서 Specification Tree를 실시간으로 숨기거나 표시하는 기능을 수행하는 전용 토글 키는 ""Ctrl + Shift + T""입니다. 📍 참조 근거: 1주차 (Basic).pdf (P.13), 1주차 (Basic).pdf (P.8)"
Q02,CATIA V5에서 'Part Design Workbench'를 즉시 활성화하기 위해 설정할 수 있는 기본 제공 가속기(Accelerator) 단축키는 무엇입니까?,Ctrl+W,"[RAG Off / Direct LLM] ⭐ Upstage Solar: 틀림 (2/5) 💬 제시된 단축키(Ctrl+Alt+F7~F9)는 Part Design Workbench 활성화와 무관하며, 실제 기본 가속기 단축키인 'Ctrl+Shift+D'를 누락한 오답입니다. CATIA V5에서 'Part Design Workbench'을 즉시 활성화하기 위해서는 기본적으로 제공하는 가속기를 사용해야 합니다. 이 가속기는 CATIA V5의 기본 환경에 따라 다르지만, 일반적으로 다음과 같은 단축키가 있습니다: 1. Ctrl + Alt + F9: 이 단축키는 CATIA V5의 기본 환경으로부터 'Part Design Workbench'을 활성화합니다. 2. Ctrl + Alt + F8: 이 단축키는 CATIA V5의 기본 환경으로부터 'Part Design Workbench'을 활성화합니다. 3. Ctrl + Alt + F7: 이 단축키는 CATIA V5의 기본 환경으로부터 'Part Design Workbench'을 활성화합니다. 이 단축키를 사용하면 CATIA V5의 기본 환경으로부터 'Part Design Workbench'을 즉시 활성화할 수 있습니다.","[RAG On / 매뉴얼 참조] ⭐ Upstage Solar: 맞음 (5/5) 💬 CATIA V5에서 Part Design Workbench를 활성화하는 기본 가속기 단축키는 정확히 'Alt + F8'로, 기능명과 단축키 모두 오류 없이 명확히 설명됨. CATIA V5에서 'Part Design Workbench'를 즉시 활성화하기 위해 설정할 수 있는 기본 제공 가속기(Accelerator) 단축키는 ""Alt + F8""입니다. 📍 참조 근거: 2주차 (Sketcher).pdf (P.7), EDU_CAT_EN_V5F_FB_V5R19.pdf (P.1)"
Q03,"CATIA V5에서 'Reference Elements (Extended)' 기능을 사용해 YZ 평면에서 100mm 오프셋된 평면을 생성할 때, Type 설정에서 On Plane 선택 후 H와 V 값을 각각 몇으로 지정해야 하는가?",H와 V 값은 모두 0으로 지정해야 합니다.,"[RAG Off / Direct LLM] ⭐ Upstage Solar: 틀림 (2/5) 💬 H와 V 값을 모두 100mm로 설정하는 것은 오답. YZ 평면 오프셋 시 H(수평) 값은 0, V(수직) 값만 100mm로 설정해야 함. YZ 평면에서 100mm의 오프셋된 평면을 생성하기 위해 `Reference Elements (Extended)` 기능을 사용하면 다음과 같은 방법으로 가능합니다: 1. **Type 설정**: `On Plane`를 선택합니다. 2. **H 값**: 100mm로 설정합니다. 3. **V 값**: 100mm로 설정합니다. 이렇게 하면 YZ 평면에서 100mm의 오프셋된 평면을 생성하게 됩니다.","[RAG On / 매뉴얼 참조] ⭐ Upstage Solar: 틀림 (2/5) 💬 H와 V 값을 0으로 지정하는 것은 오프셋 방향과 거리 설정과 무관하며, YZ 평면 오프셋 시 V 값만 100mm로 설정해야 함. 핵심 개념 오류로 오답에 가까움. CATIA V5에서 'Reference Elements (Extended)' 기능을 사용해 YZ 평면에서 100mm 오프셋된 평면을 생성하는 경우, Type 설정에서 On Plane 선택 후 H와 V 값을 각각 0, 0으로 지정하면 됩니다. 예를 들어, 이 예시에서는 H와 V 값을 각각 0으로 지정합니다: - H = 0 - V = 0 이렇게 하면 YZ 평면에서 100mm의 오프셋된 평면을 생성하게 됩니다. 📍 참조 근거: 10주차 (Part Design)_rev01.pdf (P.19), 1주차 (Basic).pdf (P.12)"
Q04,CATIA V5의 Tools 툴바에서 문서의 업데이트가 필요한 엔티티가 있을 때 활성화되는 툴의 정확한 명칭은 무엇인가요?,Update All,"[RAG Off / Direct LLM] ⭐ Upstage Solar: 전혀 아님/환각 (1/5) 💬 평가 예외 발생 CATIA V5의 Tools 툴바에서 문서 업데이트를 위한 활성화된 툴의 정확한 명칭은 ""Update"" 또는 ""Update Tool""입니다. 이 명칭은 다음과 같은 이유로 사용됩니다: 1. ""Update""는 업데이트를 의미하며, 이 명칭은 업데이트를 포함하는 작업을 나타냅니다. 2. ""Update Tool""은 업데이트를 수행하기 위해 사용할 수 있는 툴입니다. 이 두 가지 명칭 모두 CATIA V5의 Tools 툴바에서 문서 업데이트를 지원하는 활성화된 툴입니다.","[RAG On / 매뉴얼 참조] ⭐ Upstage Solar: 틀림 (2/5) 💬 질문의 핵심인 '업데이트 필요 시 활성화되는 툴'을 정확히 언급하지 못하고, Tools 툴바의 일반적인 기능들을 나열하여 오답에 가까움. CATIA V5의 Tools 툴바에서 문서의 업데이트가 필요한 엔티티가 있을 때 활성화되는 툴의 정확한 명칭은 다음과 같습니다: - **Update** : 이 엔티티는 업데이트를 위한 작업을 수행하는 데 사용됩니다. - **Save As** : 이 엔티티는 파일을 저장하거나 다른 파일 형식으로 변환할 수 있는 기능입니다. - **Close** : 이 엔티티는 문서를 닫아야 합니다. - **Print** : 이 엔티티는 문서를 출력할 수 있습니다. - **Redo** : 이 엔티티는 문서를 재작성하고 다시 실행할 수 있는 기능입니다. - **Undo** : 이 엔티티는 문서를 재작성하고 다시 실행하지 않도록 하는 기능입니다. - **New Document** : 이 엔티티는 새로운 문서를 생성합니다. - **Open** : 이 엔티티는 문서를 열기 위해 사용됩니다. - **Close All Documents** : 이 엔티티는 모든 문서를 닫아야 

## 📊 6. BERTScore 정량 평가 및 결과 CSV 저장
Solar Score Gap ≥ 2 최우수 질문 12개에 대해 **BERTScore (Precision, Recall, F1)** 및 **Upstage Solar Avg Score**를 계산하여 요약표를 출력하고 결과를 저장합니다.

In [6]:
# 6. BERTScore 정량 지표 계산 및 결과 저장
refs = df_gap2["reference_answer"].astype(str).tolist()
direct_cands = [str(a) for a in direct_answers]
rag_cands = [str(a) for a in rag_answers]

print(f"[Metric] Computing BERTScore (Precision, Recall, F1) for {MODEL_NAME}...\n")
P_dir, R_dir, F1_dir = bert_score_compute(cands=direct_cands, refs=refs, lang="ko", verbose=False)
P_rag, R_rag, F1_rag = bert_score_compute(cands=rag_cands, refs=refs, lang="ko", verbose=False)

f1_list_dir = [round(f, 4) for f in F1_dir.tolist()]
f1_list_rag = [round(f, 4) for f in F1_rag.tolist()]
avg_f1_dir = float(F1_dir.mean())
avg_f1_rag = float(F1_rag.mean())
f1_improvement = avg_f1_rag - avg_f1_dir

avg_solar_dir = sum(solar_direct_scores) / len(solar_direct_scores)
avg_solar_rag = sum(solar_rag_scores) / len(solar_rag_scores)
solar_improvement = avg_solar_rag - avg_solar_dir

# 1) 종합 요약 표
df_summary_score = pd.DataFrame([{
    "Model": MODEL_NAME,
    "Target High-Gap Questions": len(df_gap2),
    "Direct LLM Solar Avg (1~5점)": f"{avg_solar_dir:.2f} / 5.0",
    "RAG LLM Solar Avg (1~5점)": f"{avg_solar_rag:.2f} / 5.0",
    "Solar Score Improvement (+Δ점수)": f"+{solar_improvement:.2f}점",
    "Direct LLM BERT F1": f"{avg_f1_dir:.4f}",
    "RAG LLM BERT F1": f"{avg_f1_rag:.4f}",
    "BERT F1 Improvement (+ΔF1)": f"+{f1_improvement:.4f}"
}])

# 2) 질문별 세부 점수 표
df_detail_scores = pd.DataFrame({
    "ID": df_gap2["id"],
    "Question": df_gap2["question"].apply(lambda x: str(x)[:30] + "..."),
    "Direct Solar Score": [f"{s}/5" for s in solar_direct_scores],
    "RAG Solar Score": [f"{s}/5" for s in solar_rag_scores],
    "Solar ΔScore": [f"+{r - d}" if r >= d else f"{r - d}" for d, r in zip(solar_direct_scores, solar_rag_scores)],
    "Direct BERT F1": f1_list_dir,
    "RAG BERT F1": f1_list_rag,
    "BERT ΔF1": [round(r - d, 4) for d, r in zip(f1_list_dir, f1_list_rag)]
})

print("=========================================================================================")
print(f"      {MODEL_NAME} Solar Score Gap ≥ 2 High-Gap Performance Summary                     ")
print("=========================================================================================")
display(df_summary_score)
print("\n🔹 [Solar Score Gap ≥ 2 질문별 세부 BERTScore F1 & Solar Score 대조 표]")
display(df_detail_scores)

# 3) 결과 CSV 저장
out_df = df_gap2.copy()
out_df["direct_answer"] = direct_answers
out_df["rag_answer"] = rag_answers
out_df["solar_direct_score"] = solar_direct_scores
out_df["solar_direct_reason"] = solar_direct_reasons
out_df["solar_rag_score"] = solar_rag_scores
out_df["solar_rag_reason"] = solar_rag_reasons
out_df["direct_f1"] = f1_list_dir
out_df["rag_f1"] = f1_list_rag
out_df["bert_f1_gap"] = [round(r - d, 4) for d, r in zip(f1_list_dir, f1_list_rag)]

out_csv_path = PROJECT_ROOT / "eval" / "CATIA_Solar_Gap_2_Plus_Evaluation_Results.csv"
out_df.to_csv(out_csv_path, index=False, encoding="utf-8-sig")
print(f"\n💾 [Saved] Full evaluation results exported to '{out_csv_path}'")


[Metric] Computing BERTScore (Precision, Recall, F1) for Qwen 2.5 0.5B...

      Qwen 2.5 0.5B Solar Score Gap ≥ 2 High-Gap Performance Summary                     


,Model,Target High-Gap Questions,Direct LLM Solar Avg (1~5점),RAG LLM Solar Avg (1~5점),Solar Score Improvement (+Δ점수),Direct LLM BERT F1,RAG LLM BERT F1,BERT F1 Improvement (+ΔF1)
0,Qwen 2.5 0.5B,12,1.92 / 5.0,4.00 / 5.0,+2.08점,0.5787,0.6388,+0.0601



🔹 [Solar Score Gap ≥ 2 질문별 세부 BERTScore F1 & Solar Score 대조 표]


,ID,Question,Direct Solar Score,RAG Solar Score,Solar ΔScore,Direct BERT F1,RAG BERT F1,BERT ΔF1
0,Q01,CATIA V5에서 Specification Tree를...,2/5,5/5,+3,0.5275,0.6033,0.0758
1,Q02,CATIA V5에서 'Part Design Workbe...,2/5,5/5,+3,0.5437,0.5998,0.0561
2,Q03,CATIA V5에서 'Reference Elements...,2/5,2/5,+0,0.6607,0.7087,0.0480
3,Q04,CATIA V5의 Tools 툴바에서 문서의 업데이트가...,1/5,2/5,+1,0.5324,0.4976,-0.0348
4,Q05,CATIA V5에서 'Navigation Mode' 메...,2/5,4/5,+2,0.5724,0.5943,0.0219
5,Q06,CATIA V5에서 모든 툴바와 도구가 화면상에서 사라...,2/5,4/5,+2,0.7342,0.7731,0.0389
6,Q07,"CATIA V5에서 Shell 기능을 적용할 때, 특정...",1/5,3/5,+2,0.5577,0.7348,0.1771
7,Q08,CATIA V5에서 'Specification Tree...,2/5,5/5,+3,0.5350,0.6464,0.1114
8,Q09,CATIA V5에서 'View Mode' 도구 아이콘의...,1/5,5/5,+4,0.5891,0.6232,0.0341
9,Q10,CATIA V5에서 새 파트 생성 시 축 시스템(Axi...,2/5,4/5,+2,0.5941,0.6220,0.0279



💾 [Saved] Full evaluation results exported to 'C:\KDT_14\[11]Transformer\project\team-03-project\eval\CATIA_Solar_Gap_2_Plus_Evaluation_Results.csv'
